# NB01 — Injection Sanity Check

Visual validation of all 10 anomaly injectors.  
For each type we show:
- Original vs injected signal (first 2 channels)
- Per-channel z-score before / after injection
- Cross-correlation (C-types only)
- Marginal check result (must stay within 2.5σ for C-types)

**Key invariant:** C-type injections must not raise any channel's marginal z-score above the pre-injection maximum.

In [ ]:
from __future__ import annotations

import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

from src.data.generator import VARGenerator
from src.taxonomy.types import AnomalyType, ANOMALY_REGISTRY, marginal_check_report
from src.taxonomy.injectors import INJECTOR_REGISTRY

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Base signal ──────────────────────────────────────────────────────────────
SEED = 42
T = 600
C = 4
gen = VARGenerator(n_channels=C, lag_order=2, T=T, spectral_radius=0.7, seed=SEED)
x_base, meta = gen.generate()

# Injection window used across all types
WIN_START, WIN_END = 200, 350
window = (WIN_START, WIN_END)

print(f'Signal shape: {x_base.shape}')
print(f'Injection window: [{WIN_START}, {WIN_END})')
print(f'Channels: {C}')

In [ ]:
def plot_injection(atype: AnomalyType, x_orig: np.ndarray, x_inj: np.ndarray,
                   window: tuple, show_xcorr: bool = False):
    """Plot original vs injected signal with marginal check summary."""
    T, C = x_orig.shape
    s, e = window
    meta_info = ANOMALY_REGISTRY[atype]

    report = marginal_check_report(x_orig, x_inj, window)
    passed = report['passed']
    status_color = 'green' if passed else 'red'
    status_label = 'PASS' if passed else 'FAIL'

    n_rows = 3 if show_xcorr else 2
    fig, axes = plt.subplots(n_rows, C, figsize=(4 * C, 2.5 * n_rows),
                              constrained_layout=True)
    if C == 1:
        axes = axes.reshape(n_rows, 1)

    t = np.arange(T)
    title = (f'{atype.value} — {meta_info.name}  '
             f'[{meta_info.layer.value}]  '
             f'Marginal check: [{status_label}]')
    fig.suptitle(title, fontsize=11, color=status_color, fontweight='bold')

    for c in range(C):
        ax = axes[0, c]
        ax.plot(t, x_orig[:, c], lw=0.8, color='steelblue', alpha=0.8, label='original')
        ax.plot(t, x_inj[:, c], lw=0.8, color='tomato', alpha=0.8, label='injected')
        ax.axvspan(s, e, color='orange', alpha=0.15, label='anomaly window')
        ax.set_title(f'ch {c}', fontsize=9)
        if c == 0:
            ax.set_ylabel('value')
            ax.legend(fontsize=7, loc='upper left')

        # z-score row
        ax2 = axes[1, c]
        mu, sigma = x_orig[:, c].mean(), x_orig[:, c].std() or 1.0
        z_orig = np.abs((x_orig[:, c] - mu) / sigma)
        z_inj  = np.abs((x_inj[:, c]  - mu) / sigma)
        ax2.plot(t, z_orig, lw=0.7, color='steelblue', alpha=0.7)
        ax2.plot(t, z_inj,  lw=0.7, color='tomato',    alpha=0.7)
        ax2.axhline(2.5, color='gray', ls='--', lw=0.8, label='2.5σ')
        ax2.axvspan(s, e, color='orange', alpha=0.15)
        ax2.set_ylim(bottom=0)
        if c == 0:
            ax2.set_ylabel('|z-score|')
            ax2.legend(fontsize=7)

        # cross-correlation row (for C-types)
        if show_xcorr and C > 1 and n_rows == 3:
            ax3 = axes[2, c]
            if c < C - 1:
                from scipy.signal import correlate
                lags = np.arange(-30, 31)
                center = len(x_orig[:, c]) - 1
                for col, (arr, label, color) in enumerate([
                    (x_orig, 'orig', 'steelblue'), (x_inj, 'inj', 'tomato')
                ]):
                    xi = arr[:, c] - arr[:, c].mean()
                    xj = arr[:, c+1] - arr[:, c+1].mean()
                    si = xi.std() or 1.0
                    sj = xj.std() or 1.0
                    full_ccf = correlate(xi, xj, mode='full') / (T * si * sj)
                    ccf_slice = full_ccf[center - 30: center + 31]
                    ax3.plot(lags, ccf_slice, lw=0.8, color=color, alpha=0.8, label=label)
                ax3.axhline(0, color='gray', lw=0.5)
                ax3.set_xlabel('lag')
                if c == 0:
                    ax3.set_ylabel('CCF')
                    ax3.legend(fontsize=7)
                ax3.set_title(f'CCF ch{c}↔ch{c+1}', fontsize=9)
            else:
                ax3 = axes[2, c]
                ax3.axis('off')

    # Print per-channel z-score summary
    max_z_orig = report['per_channel_orig_max_z']
    max_z_inj  = report['per_channel_inj_max_z']
    print(f'  {atype.value}: {status_label}  '
          + '  '.join(f'ch{c}: orig={max_z_orig[c]:.2f}→inj={max_z_inj[c]:.2f}'
                      for c in range(C)))
    plt.show()

print('plot_injection() defined')

---
## A-type injectors (Marginal-visible)

In [ ]:
# A1 — Global Point
import inspect

def _make_injector(atype: AnomalyType, seed: int = 42):
    cls = INJECTOR_REGISTRY[atype]
    sig = inspect.signature(cls.__init__)
    kwargs = {'seed': seed} if 'seed' in sig.parameters else {}
    return cls(**kwargs)

for atype in [AnomalyType.A1, AnomalyType.A2]:
    injector = _make_injector(atype)
    x_inj = injector.inject(x_base.copy(), window)
    plot_injection(atype, x_base, x_inj, window, show_xcorr=False)

---
## B-type injectors (Temporal-context)

In [ ]:
for atype in [AnomalyType.B1, AnomalyType.B2, AnomalyType.B3]:
    injector = _make_injector(atype)
    x_inj = injector.inject(x_base.copy(), window)
    plot_injection(atype, x_base, x_inj, window, show_xcorr=False)

---
## C-type injectors (Inter-metric, joint-only-visible)

Critical invariant: **each channel's z-score must not increase beyond the pre-injection maximum**.  
The cross-correlation row shows how the injector disrupts inter-channel structure while preserving marginals.

In [ ]:
for atype in [AnomalyType.C1, AnomalyType.C2, AnomalyType.C3, AnomalyType.C4, AnomalyType.C5]:
    injector = _make_injector(atype, seed=SEED)
    x_inj = injector.inject(x_base.copy(), window)
    plot_injection(atype, x_base, x_inj, window, show_xcorr=True)

---
## Marginal invariant: quantitative summary across all 10 types

In [ ]:
import pandas as pd

rows = []
for atype in AnomalyType:
    injector = _make_injector(atype, seed=SEED)
    x_inj = injector.inject(x_base.copy(), window)
    report = marginal_check_report(x_orig=x_base, x_injected=x_inj, window=window)
    max_orig = max(report['per_channel_orig_max_z'])
    max_inj  = max(report['per_channel_inj_max_z'])
    rows.append({
        'type': atype.value,
        'layer': ANOMALY_REGISTRY[atype].layer.value,
        'marginal_check': '✓ PASS' if report['passed'] else '✗ FAIL',
        'max_z_orig': round(max_orig, 3),
        'max_z_inj':  round(max_inj, 3),
        'delta_z': round(max_inj - max_orig, 3),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f'\nAll passed: {(df.marginal_check == "✓ PASS").all()}')

---
## C1 — IAAFT surrogate: power-spectrum preservation check

The IAAFT surrogate preserves the marginal distribution AND the power spectrum of the original channel.  
Here we verify that the periodogram of the injected channel closely matches the original.

In [ ]:
from src.taxonomy.injectors import C1CorrelationBreakInjector

inj_c1 = C1CorrelationBreakInjector(seed=SEED, n_iter=32)
x_c1 = inj_c1.inject(x_base.copy(), window)

s, e = window
ch = 0   # channel that gets phase-randomised

seg_orig = x_base[s:e, ch]
seg_inj  = x_c1[s:e, ch]

freqs = np.fft.rfftfreq(len(seg_orig))
psd_orig = np.abs(np.fft.rfft(seg_orig))**2
psd_inj  = np.abs(np.fft.rfft(seg_inj))**2

fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)

axes[0].plot(freqs, psd_orig, lw=0.9, color='steelblue', label='original')
axes[0].plot(freqs, psd_inj,  lw=0.9, color='tomato', alpha=0.8, label='injected (IAAFT)')
axes[0].set_xlabel('frequency')
axes[0].set_ylabel('PSD')
axes[0].set_title('Power spectrum: original vs IAAFT surrogate (ch 0)')
axes[0].legend()

# Sorted values should match (rank-adjust guarantees exact marginal)
axes[1].scatter(np.sort(seg_orig), np.sort(seg_inj), s=4, alpha=0.7, color='purple')
lo, hi = seg_orig.min(), seg_orig.max()
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=0.8)
axes[1].set_xlabel('sorted original values')
axes[1].set_ylabel('sorted injected values')
axes[1].set_title('Quantile-quantile: injected == original (rank-adjust)')
plt.show()

max_qq_dev = np.max(np.abs(np.sort(seg_orig) - np.sort(seg_inj)))
print(f'Max Q-Q deviation (should be 0 with rank-adjust): {max_qq_dev:.6f}')

---
## C1 — Cross-correlation breakdown

C1 should destroy the cross-correlation between the phase-randomised channel and all others,  
while keeping the auto-correlation (marginal temporal structure) intact.

In [ ]:
from scipy.signal import correlate

s, e = window
pairs = [(0, 1), (0, 2), (1, 2)]

fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 3), constrained_layout=True)
lags = np.arange(-25, 26)

for ax, (i, j) in zip(axes, pairs):
    for arr, label, color in [(x_base, 'original', 'steelblue'), (x_c1, 'C1 injected', 'tomato')]:
        seg = arr[s:e]
        xi = seg[:, i] - seg[:, i].mean()
        xj = seg[:, j] - seg[:, j].mean()
        si = xi.std() or 1.0
        sj = xj.std() or 1.0
        L = len(xi)
        full_ccf = correlate(xi, xj, mode='full') / (L * si * sj)
        center = L - 1
        ccf_slice = full_ccf[center - 25: center + 26]
        ax.plot(lags, ccf_slice, lw=1.0, color=color, alpha=0.85, label=label)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title(f'CCF ch{i}↔ch{j} (inside window)')
    ax.set_xlabel('lag')
    if i == 0 and j == 1:
        ax.legend(fontsize=8)

fig.suptitle('C1 injection: cross-correlation inside anomaly window', fontsize=11)
plt.show()

---
## Multi-window injection: no overlap guarantee

In [ ]:
# Simulate the multi-window injection used in experiments
from src.taxonomy.injectors import C1CorrelationBreakInjector

n_windows = 5
win_size  = 80
spacing   = T // (n_windows + 1)
starts    = [spacing * (i + 1) for i in range(n_windows)]

injector = C1CorrelationBreakInjector(seed=SEED)
x_multi = x_base.copy()
labels   = np.zeros(T, dtype=int)

for s in starts:
    e = min(s + win_size, T)
    x_multi = injector.inject(x_multi, (s, e))
    labels[s:e] = 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 4), constrained_layout=True)
t = np.arange(T)
ax1.plot(t, x_base[:, 0], lw=0.8, color='steelblue', label='original ch0')
ax1.plot(t, x_multi[:, 0], lw=0.8, color='tomato', alpha=0.8, label='multi-inject ch0')
for s in starts:
    ax1.axvspan(s, min(s + win_size, T), color='orange', alpha=0.2)
ax1.set_ylabel('value')
ax1.legend(fontsize=8)
ax1.set_title(f'Multi-window C1 injection: {n_windows} windows × {win_size} steps')

ax2.fill_between(t, labels, alpha=0.5, color='orange', label='anomaly label')
ax2.set_ylim(-0.1, 1.3)
ax2.set_ylabel('label')
ax2.set_xlabel('timestep')
ax2.legend(fontsize=8)
plt.show()

print(f'Total anomalous timesteps: {labels.sum()} / {T}')
print(f'Anomaly ratio: {labels.mean():.2%}')